# 2. Entra ID, Authentication Methods & Access Management

This notebook covers the meat of the Entra domain: identity types, authentication methods (MFA, passwordless), Conditional Access, and RBAC.

## Microsoft Entra ID — what it actually is

Entra ID (formerly Azure AD) is Microsoft's cloud-based identity and access management service. It's the **IdP** for:
- Azure portal and Azure resources
- Microsoft 365 (Outlook, Teams, SharePoint)
- Thousands of third-party SaaS apps (Salesforce, ServiceNow, etc.)
- Your own custom applications

Every Azure subscription is associated with an Entra ID **tenant** (directory).

---
## Identity types in Entra ID

| Type | What it is | Example |
|------|-----------|----------|
| **User** | A person (employee, contractor) | alice@contoso.com |
| **Guest user** | External person invited via B2B (Microsoft Entra External ID) | bob@fabrikam.com |
| **Service principal** | An app's identity in the tenant | API-A's service principal |
| **Managed identity** | Azure-managed SP (no secrets) | Container App's identity |
| **Agent identity (Microsoft Entra Agent ID)** | Identity for an AI agent, with a human **sponsor** accountable for it | A Copilot agent that reads SharePoint |
| **Device** | A registered/joined computer | Alice's laptop |
| **Group** | Collection of users/devices | "Marketing team" group |

*Service principals and managed identities together are called **workload identities** — the
non-human half of the directory. As of the July 2026 exam update, **agent ID** is called out
explicitly in the study guide: an AI agent gets its own identity so it can be signed in,
Conditional-Access'd, risk-scored and access-reviewed like any other principal.*

### Hybrid identity

Most enterprises have on-prem AD *and* Entra ID. **Hybrid identity** means the same person has
one identity across both. The sync tool and the sign-in method are two separate choices:

| Sync tool | How it works |
|-----------|--------------|
| **Microsoft Entra Connect** (formerly Azure AD Connect) | A server you run on-premises that syncs users/groups from AD DS to Entra ID |
| **Microsoft Entra Cloud Sync** | A lightweight cloud-managed agent; Microsoft's stated replacement for Entra Connect |

| Sign-in method | Where the password is actually checked |
|----------------|---------------------------------------|
| **Password hash sync (PHS)** | In the cloud. A hash *of* the on-prem password hash is synced to Entra ID |
| **Pass-through authentication (PTA)** | On-premises. An agent validates against AD DS in real time; no hashes stored in the cloud |
| **Federation (AD FS)** | On-premises AD FS authenticates; Entra ID trusts the token it issues |

**Exam tip**: PHS is the simplest and the one Microsoft recommends — it also survives an
on-prem outage, and it is what lets ID Protection detect *leaked credentials*. Federation is
the most complex: more servers, more certificates, more to break.

---
## App registrations, enterprise apps, service principals, managed identities

Beginners find this confusing because it looks like four names for the same thing. Here's the mental model:

- **App registration** = the *definition* of an app (name, redirect URIs, allowed scopes, secrets). It lives in the tenant where the developer built the app. One per app.
- **Service principal (Enterprise application)** = the *local instance* of an app in **your** tenant. It's what gets role assignments and consent. You get one automatically when an app is registered in your tenant, or when a multi-tenant app is consented to.
- **Managed identity** = a special service principal that Azure **creates and rotates secrets for you** — used by Azure resources (VMs, Functions, Container Apps) to call other Azure resources. No secret to leak.
  - **System-assigned** = tied to the lifecycle of one resource, deleted with it.
  - **User-assigned** = a standalone resource that can be attached to many resources.

### Analogy

Think of the app registration as a *blueprint* and each service principal as a *house built from that blueprint* in a specific neighborhood (tenant). A managed identity is a house where Azure handles the keys for you so you never have to carry them.

**Exam takeaway**: if a question mentions "no secrets in code" or "an Azure resource calling another Azure resource", the answer is almost always **managed identity**.

---
## Authentication methods

Entra ID supports many ways to prove your identity. What the exam actually tests is the
*ordering* — and the fact that **"is it MFA?"** and **"is it phishing-resistant?"** are two
different questions.

| Method | Factor(s) | Phishing-resistant? | Notes |
|--------|-----------|---------------------|-------|
| Password | know | ❌ | Reusable, guessable, sprayed at scale |
| Security questions | know | ❌ | SSPR only — never a second factor for sign-in |
| SMS or voice-call OTP | have (weakly) | ❌ | **The weakest second factor.** SIM swap, SS7 interception, and a code the user can simply read out to a caller. Better than a password alone; Microsoft steers you off it |
| Email OTP | have | ❌ | SSPR and guest sign-in, not workforce MFA |
| OATH TOTP code (Authenticator or hardware token) | have | ❌ | The code can be typed straight into an attacker's proxy page |
| Authenticator push + number matching | have | ❌ | Number matching kills MFA-fatigue prompt bombing, but an adversary-in-the-middle proxy still relays it |
| Certificate-based authentication (CBA) | have | ✅ | |
| Passkey (FIDO2) — security key, or passkey in Microsoft Authenticator | have, unlocked by know/are | ✅ | |
| Windows Hello for Business | have (device-bound key) + know/are (PIN or biometric) | ✅ | Multifactor on its own |

**Phishing-resistant** means the credential is cryptographically bound to the real site, so a
pixel-perfect fake sign-in page gets nothing it can replay. Microsoft's phishing-resistant list
is: Windows Hello for Business, Platform Credential for macOS, passkeys (FIDO2 security keys and
passkeys in Microsoft Authenticator), and certificate-based authentication.

**Naming**: "passkey (FIDO2)" is the current Microsoft term; older material says "FIDO2 security
key". And Windows Hello for Business is not simply "something you are" — the biometric or PIN
unlocks a private key that never leaves the device's TPM, which is why one gesture counts as
two factors.

### MFA (Multi-Factor Authentication)

MFA requires **two or more** factors from *different* categories:
- Something you **know** (password, PIN)
- Something you **have** (phone, security key, device-bound key)
- Something you **are** (fingerprint, face)

**Exam fact**: Microsoft's headline figure is that an account is *more than 99.9% less likely to
be compromised* when MFA is switched on. A later Microsoft measurement study put the reduction
at 99.2% across the population, and 98.6% even when the password had already leaked. Either
number makes the same point: MFA is the single highest-value identity control. It is not
absolute — what gets through is adversary-in-the-middle proxies and token theft, which is
exactly what phishing-resistant methods close off.

### Password protection and management

| Capability | What it does |
|------------|--------------|
| **Self-Service Password Reset (SSPR)** | Users reset their own password using pre-registered methods (phone, email, security questions). Cuts helpdesk calls. You can require 1 or 2 methods |
| **Global banned password list** | Microsoft-maintained list of weak base terms, on for every tenant, cannot be disabled or viewed |
| **Custom banned password list** | Up to 1,000 org-specific terms — brand, product, HQ city. Requires P1/P2 |
| **Entra Password Protection for AD DS** | Agents on your domain controllers apply the same lists to on-prem password changes |
| **Smart lockout** | Locks out an attacker guessing at an account while letting the real user, on a familiar device, keep signing in |

The banned-list check is deliberately *fuzzy*: it normalises the password first
(`Bl@nK` → `blank`) and then matches within an edit distance of one, so banning the base term
also blocks its variants. Ban `Contoso`, not `Contoso!1`.

In [ ]:
import json

# Each factor maps to the category (or categories) it satisfies, plus whether the
# credential is phishing-resistant — bound to the real site so a proxy cannot replay it.
CATEGORIES = {
    'password':           {'know'},
    'security_question':  {'know'},
    'sms_otp':            {'have'},
    'authenticator_push': {'have'},
    'totp_code':          {'have'},
    'fido2_passkey':      {'have', 'are'},  # passkey with user verification: the
                                            # authenticator you hold + the biometric/PIN
                                            # that unlocks its private key
    'certificate':        {'have'},
    'windows_hello':      {'have', 'are'},  # device-bound key unlocked by PIN/biometric
    'fingerprint':        {'are'},
    'face_recognition':   {'are'},
}
PHISHING_RESISTANT = {'fido2_passkey', 'certificate', 'windows_hello'}
LABEL = {'know': 'something you know', 'have': 'something you have', 'are': 'something you are'}

def evaluate_mfa(factors: list[str]) -> dict:
    used = set()
    details = []
    for f in factors:
        cats = CATEGORIES.get(f, set())
        used |= cats
        details.append({'factor': f, 'categories': sorted(LABEL[c] for c in cats) or ['unknown']})

    is_mfa = len(used) >= 2
    # Entra models this the same way its "phishing-resistant MFA" authentication strength does:
    # the whole sign-in has to be satisfied by phishing-resistant credentials.
    resistant = bool(factors) and all(f in PHISHING_RESISTANT for f in factors)
    return {
        'factors': details,
        'unique_categories': sorted(LABEL[c] for c in used),
        'is_multi_factor': is_mfa,
        'is_phishing_resistant': resistant,
        'verdict': ('✅ MFA satisfied' if is_mfa else '❌ Single factor only')
                   + (' — and phishing-resistant' if resistant else ' — phishable'),
    }

for title, factors in [
    ('Scenario 1: password only', ['password']),
    ('Scenario 2: password + SMS OTP', ['password', 'sms_otp']),
    ('Scenario 3: password + security question (both "know")', ['password', 'security_question']),
    ('Scenario 4: passkey (FIDO2) on its own — one gesture, two factors',
     ['fido2_passkey']),
    ('Scenario 5: Windows Hello for Business on its own', ['windows_hello']),
]:
    print(f'=== {title} ===')
    print(json.dumps(evaluate_mfa(factors), indent=2))
    print()

# The lesson only holds if the simulation keeps drawing these distinctions.
assert evaluate_mfa(['password'])['is_multi_factor'] is False
assert evaluate_mfa(['password', 'security_question'])['is_multi_factor'] is False, \
    'two "something you know" factors are not MFA'
assert evaluate_mfa(['password', 'sms_otp'])['is_multi_factor'] is True
assert evaluate_mfa(['password', 'sms_otp'])['is_phishing_resistant'] is False, \
    'SMS is a phishable second factor — MFA is not the same claim as phishing-resistant'
assert evaluate_mfa(['password', 'authenticator_push'])['is_phishing_resistant'] is False, \
    'push approval is relayable by an adversary-in-the-middle proxy'
assert evaluate_mfa(['fido2_passkey'])['is_multi_factor'] is True, \
    'a passkey with user verification is possession + biometric/PIN — MFA in one gesture'
assert evaluate_mfa(['fido2_passkey'])['is_phishing_resistant'] is True
assert evaluate_mfa(['windows_hello'])['is_multi_factor'] is True, \
    'Windows Hello for Business is multifactor on its own (device-bound key + PIN/biometric)'
print('✅ MFA / phishing-resistance distinctions hold')

Notice scenario 3: password + security question = **NOT MFA** because both are "something you know". The exam tests this.

---
## Conditional Access

Conditional Access is Microsoft's **Zero Trust policy engine**. It evaluates signals and enforces decisions:

```
IF [conditions met]  →  THEN [grant/block/require MFA]
```

Zero Trust has three guiding principles, and the exam wants them by name:

| Principle | Meaning | How Conditional Access expresses it |
|-----------|---------|-------------------------------------|
| **Verify explicitly** | Always authenticate and authorise from all available signals | Evaluate user, device, location, app and risk on *every* request |
| **Use least privilege** | Just-enough access, just-in-time, risk-based adaptive policy | Grant the narrowest role; elevate admin roles through PIM |
| **Assume breach** | Segment, verify end to end, minimise blast radius | Block high risk, require a compliant device, restrict the session |

Note the ordering: Conditional Access runs **after** first-factor authentication. It is a policy
gate on an already-identified principal, not a front door against volumetric attacks.

### Signals (conditions)

| Signal | Example |
|--------|---------|
| User/group membership | "All users in Sales group" |
| Location (IP/named location) | "Not from trusted office IPs" |
| Device platform | "iOS devices only" |
| Device compliance | "Must be Intune-compliant" |
| Application | "When accessing SharePoint" |
| Risk level | "Sign-in risk is medium or high" (Identity Protection) |
| Client app | "Legacy authentication clients" |

### Decisions

| Decision | What happens |
|----------|--------------|
| **Block** | Access denied, period |
| **Grant** | Access allowed |
| **Grant + require MFA** | Allowed only if MFA is completed |
| **Grant + require compliant device** | Allowed only from managed devices |
| **Grant + require approved app** | Only certain apps can access the resource |
| **Session controls** | Limited session (e.g., can't download files) |

In [ ]:
from dataclasses import dataclass

@dataclass
class SignIn:
    user: str
    group: str
    app: str
    location: str
    device_compliant: bool
    mfa_done: bool
    risk_level: str  # 'none', 'low', 'medium', 'high'

# Conditional Access policies (like you'd configure in the Entra portal)
CA_POLICIES = [
    {
        'name': 'Require MFA for all users',
        'applies_to': lambda s: True,
        'grant': lambda s: 'allow' if s.mfa_done else 'require_mfa',
    },
    {
        'name': 'Block legacy auth',
        'applies_to': lambda s: s.app == 'legacy-smtp',
        'grant': lambda s: 'block',
    },
    {
        'name': 'Require compliant device for admin portal',
        'applies_to': lambda s: s.app == 'azure-portal' and s.group == 'admins',
        'grant': lambda s: 'allow' if s.device_compliant else 'block',
    },
    {
        'name': 'Block high-risk sign-ins',
        'applies_to': lambda s: s.risk_level in ('medium', 'high'),
        'grant': lambda s: 'block' if s.risk_level == 'high' else ('allow' if s.mfa_done else 'require_mfa'),
    },
]

def evaluate_conditional_access(signin: SignIn) -> dict:
    results = []
    for policy in CA_POLICIES:
        if policy['applies_to'](signin):
            decision = policy['grant'](signin)
            results.append({'policy': policy['name'], 'decision': decision})
    # Policies are additive and the most restrictive result wins: any block is a block.
    if any(r['decision'] == 'block' for r in results):
        final = '🚫 BLOCKED'
    elif any(r['decision'] == 'require_mfa' for r in results):
        final = '🔐 REQUIRES MFA'
    else:
        final = '✅ ALLOWED'
    return {'policies_evaluated': results, 'final_decision': final}

scenarios = [
    ('Normal sign-in with MFA',
     SignIn('alice', 'users', 'outlook', 'office', True, True, 'none'), '✅ ALLOWED'),
    ('Sign-in without MFA',
     SignIn('alice', 'users', 'outlook', 'office', True, False, 'none'), '🔐 REQUIRES MFA'),
    ('Admin accessing portal from non-compliant device',
     SignIn('bob', 'admins', 'azure-portal', 'office', False, True, 'none'), '🚫 BLOCKED'),
    ('High-risk sign-in from unknown location',
     SignIn('alice', 'users', 'outlook', 'unknown', True, True, 'high'), '🚫 BLOCKED'),
    ('Legacy SMTP client',
     SignIn('alice', 'users', 'legacy-smtp', 'office', True, True, 'none'), '🚫 BLOCKED'),
]

for name, signin, expected in scenarios:
    result = evaluate_conditional_access(signin)
    print(f'=== {name} ===')
    for p in result['policies_evaluated']:
        print(f'  📋 {p["policy"]}: {p["decision"]}')
    print(f'  → {result["final_decision"]}\n')
    assert result['final_decision'] == expected, \
        f'{name}: expected {expected}, got {result["final_decision"]}'

# The completed-MFA sign-in must still get through, or the policy set is just "deny all".
assert evaluate_conditional_access(scenarios[0][1])['final_decision'] == '✅ ALLOWED'
print('✅ every scenario landed on its expected decision')

---
## Conditional Access — from bad practice to Zero Trust

Let's see the same user, Alice, try to sign in under progressively stronger policies. This is the real bad→best journey most organizations go through.

In [ ]:
# Reuse the SignIn + policy engine defined above, but show different *policy sets*.

def run_policies(policies, scenarios):
    """Evaluate one policy set and return {scenario name: final decision}."""
    finals = {}
    for name, si in scenarios:
        decisions = [p['grant'](si) for p in policies if p['applies_to'](si)]
        if 'block' in decisions: final = '🚫 BLOCKED'
        elif 'require_mfa' in decisions: final = '🔐 REQUIRES MFA'
        else: final = '✅ ALLOWED'
        finals[name] = final
        print(f'  {name}: {final}')
    return finals

risky = SignIn('alice','users','outlook','unknown-country', True, False, 'high')
normal = SignIn('alice','users','outlook','office', True, True, 'none')
NORMAL, RISKY = 'Normal sign-in', 'Risky sign-in (no MFA, high risk)'
scenarios=[(NORMAL, normal), (RISKY, risky)]

print('❌ Stage 1 — No Conditional Access at all (passwords only)')
stage1 = run_policies([], scenarios)

print('\n🟡 Stage 2 — Require MFA for everyone (good baseline)')
stage2 = run_policies([
    {'name':'MFA for all','applies_to':lambda s: True,
     'grant':lambda s: 'allow' if s.mfa_done else 'require_mfa'},
], scenarios)

print('\n🟢 Stage 3 — Zero Trust (MFA + block high risk + block legacy auth)')
stage3 = run_policies([
    {'name':'MFA for all','applies_to':lambda s: True,
     'grant':lambda s: 'allow' if s.mfa_done else 'require_mfa'},
    {'name':'Block high risk','applies_to':lambda s: s.risk_level=='high',
     'grant':lambda s: 'block'},
    {'name':'Block legacy auth','applies_to':lambda s: s.app=='legacy-smtp',
     'grant':lambda s: 'block'},
], scenarios)

# The lesson is the *progression*. Assert it, so the notebook fails loudly if a
# future edit flattens the stages into "everything is allowed" or "everything is blocked".
assert stage1[RISKY] == '✅ ALLOWED', 'Stage 1 must actually let the risky sign-in through'
assert stage2[RISKY] == '🔐 REQUIRES MFA', 'Stage 2 must challenge, not block'
assert stage3[RISKY] == '🚫 BLOCKED', 'Stage 3 must block the risky sign-in'
assert stage1[NORMAL] == stage2[NORMAL] == stage3[NORMAL] == '✅ ALLOWED', \
    'tightening policy must not break the legitimate, MFA-completed user'

print('\nLesson: each stage raises the floor. A risky sign-in that Stage 1 allows gets '
      'MFA-challenged at Stage 2 and outright blocked at Stage 3 — while the normal user '
      'is untouched at every stage.')

### Key exam points on Conditional Access

- Requires **Microsoft Entra ID P1** (at minimum). The *risk* conditions — sign-in risk and
  user risk — come from ID Protection and therefore need **P2**.
- **Security defaults** are the free, all-or-nothing alternative for tenants without P1. You
  cannot run security defaults and Conditional Access policies at the same time.
- Policies are **additive** — all applicable policies must be satisfied, and **block wins**.
- "Report-only" mode lets you test policies before enforcing them.
- **Named locations** let you trust specific IP ranges (e.g., your office).
- Conditional Access is evaluated **after** the first factor, so it cannot be your front line
  against denial-of-service — but it can use those events as signals.

---
## RBAC (Role-Based Access Control)

RBAC assigns permissions through **roles** rather than granting access to individual users. Entra ID has built-in roles:

| Role | What it can do |
|------|-----------|
| **Global Administrator** | Full access to everything in the tenant |
| **User Administrator** | Manage users and groups |
| **Security Administrator** | Manage security features (Conditional Access, Identity Protection) |
| **Security Reader** | Read security info but can't change anything |
| **Billing Administrator** | Manage subscriptions and billing |
| **Application Administrator** | Manage app registrations and enterprise apps |

There are also **Azure RBAC roles** for resource access:

| Role | Scope |
|------|-----------|
| **Owner** | Full access + can assign roles |
| **Contributor** | Full access but can't assign roles |
| **Reader** | View only |
| **User Access Administrator** | Manage user access to resources |

### Exam tip

- Entra ID roles → manage the **directory** (users, groups, apps).
- Azure RBAC roles → manage **Azure resources** (VMs, storage, etc.).
- **Least privilege**: always assign the narrowest role that works.

In [ ]:
# Simulate RBAC
ROLE_PERMISSIONS = {
    'Owner':       {'read', 'write', 'delete', 'assign_roles'},
    'Contributor': {'read', 'write', 'delete'},
    'Reader':      {'read'},
}

ROLE_ASSIGNMENTS = {
    'alice': {'role': 'Owner',       'scope': '/subscriptions/123/resourceGroups/prod'},
    'bob':   {'role': 'Contributor', 'scope': '/subscriptions/123/resourceGroups/prod'},
    'carol': {'role': 'Reader',      'scope': '/subscriptions/123/resourceGroups/prod'},
}

def check_access(user: str, action: str) -> str:
    assignment = ROLE_ASSIGNMENTS.get(user)
    if not assignment:
        return f'❌ {user}: no role assigned → access denied'
    perms = ROLE_PERMISSIONS[assignment['role']]
    if action in perms:
        return f'✅ {user} ({assignment["role"]}): {action} → allowed'
    return f'❌ {user} ({assignment["role"]}): {action} → denied (not in {assignment["role"]} permissions)'

actions = ['read', 'write', 'delete', 'assign_roles']
for user in ['alice', 'bob', 'carol', 'dave']:
    for action in actions:
        print(check_access(user, action))
    print()

# Least privilege has to be observable, not just asserted in prose.
assert 'assign_roles' not in ROLE_PERMISSIONS['Contributor'], \
    'Contributor must not be able to hand out roles — that is what separates it from Owner'
assert ROLE_PERMISSIONS['Reader'] == {'read'}, 'Reader must be read-only'
assert ROLE_PERMISSIONS['Reader'] < ROLE_PERMISSIONS['Contributor'] < ROLE_PERMISSIONS['Owner'], \
    'the built-in roles should nest: Reader ⊂ Contributor ⊂ Owner'
assert 'denied' in check_access('dave', 'read'), 'no assignment means no access, not default read'
print('✅ role hierarchy holds: Reader ⊂ Contributor ⊂ Owner, and dave has nothing')

---
## Summary

| Concept | Key fact |
|---------|----------|
| **Identity types** | Users, guests (B2B), service principals, managed identities, **agent IDs**, devices, groups |
| **Hybrid identity** | Entra Connect (or Cloud Sync) copies AD → Entra ID. PHS is simplest; AD FS is the most complex. |
| **MFA** | ≥2 factors from different categories. >99.9% less likely to be compromised. |
| **Phishing-resistant** | Passkeys (FIDO2), Windows Hello for Business, CBA. SMS and push are MFA but *phishable*. |
| **SSPR** | Users reset their own passwords. Reduces helpdesk load. |
| **Password protection** | Global banned list (all tenants) + custom banned list (P1/P2), extendable to AD DS. |
| **Conditional Access** | IF signals → THEN grant/block/require MFA. Needs P1; risk conditions need P2. |
| **Zero Trust** | Verify explicitly · use least privilege · assume breach. |
| **Entra roles** | Manage the directory (Global Admin, User Admin, etc.) |
| **Azure RBAC** | Manage Azure resources (Owner, Contributor, Reader) |

---
## Self-check — Entra ID, authentication & access management

Edit `MY_ANSWERS`, re-run, and read the explanations for anything you missed.

In [ ]:
QUIZ = [
    {'id': 'Q1',
     'q': 'A user signs in with a password and an SMS code. Which statement is true?',
     'options': {'A': 'It is not MFA — both are "something you have"',
                 'B': 'It is MFA, and it is phishing-resistant',
                 'C': 'It is MFA, but it is not phishing-resistant',
                 'D': 'It is neither MFA nor phishing-resistant'},
     'a': 'C',
     'why': 'Password (know) + SMS code (have) is two categories, so it is MFA. But the code can be '
            'relayed through an attacker proxy, and the number is exposed to SIM swap, so it is not '
            'phishing-resistant. Phishing-resistant means passkeys (FIDO2), Windows Hello for Business, '
            'Platform Credential for macOS, or certificate-based authentication.'},
    {'id': 'Q2',
     'q': 'An Azure Function must read a Key Vault secret with no credentials stored anywhere. What do you use?',
     'options': {'A': 'An app registration with a client secret',
                 'B': 'A system-assigned managed identity',
                 'C': 'A service account user with a strong password',
                 'D': 'A shared access signature'},
     'a': 'B',
     'why': 'A managed identity is a service principal whose credentials Azure creates and rotates. '
            'System-assigned ties it to that one resource and deletes with it; user-assigned is a '
            'standalone object you can attach to several resources. "No secrets in code" is the tell.'},
    {'id': 'Q3',
     'q': 'Which hybrid sign-in method validates the password against on-premises AD in real time and '
          'stores no password hashes in the cloud?',
     'options': {'A': 'Password hash synchronisation', 'B': 'Pass-through authentication',
                 'C': 'Federation with AD FS', 'D': 'Microsoft Entra Cloud Sync'},
     'a': 'B',
     'why': 'PTA installs a lightweight agent that checks the password against AD DS on each sign-in. '
            'PHS syncs a hash of the hash to the cloud. AD FS also authenticates on-prem but is a full '
            'federation server farm, and Cloud Sync is a *sync tool*, not a sign-in method.'},
    {'id': 'Q4',
     'q': 'You need "block legacy authentication, and require MFA when signing in from outside the '
          'office". What feature, and what minimum licence?',
     'options': {'A': 'Security defaults, free', 'B': 'Conditional Access, Entra ID P1',
                 'C': 'Conditional Access, Entra ID P2', 'D': 'Azure RBAC, free'},
     'a': 'B',
     'why': 'Conditional Access needs at least P1. P2 is only required once a *risk* condition '
            '(sign-in risk / user risk from ID Protection) is involved. Security defaults are free but '
            'all-or-nothing — no named locations, no per-app targeting.'},
    {'id': 'Q5',
     'q': 'Which Zero Trust principle is most directly expressed by giving admins eligible-not-active '
          'roles they activate for four hours?',
     'options': {'A': 'Verify explicitly', 'B': 'Use least privilege',
                 'C': 'Assume breach', 'D': 'Defence in depth'},
     'a': 'B',
     'why': 'Least privilege is stated by Microsoft as "just-in-time and just-enough-access" — exactly '
            'what PIM does. Verify explicitly is about authenticating on all signals; assume breach is '
            'about segmentation and blast radius. Defence in depth is not one of the three principles.'},
    {'id': 'Q6',
     'q': 'Bob needs to create and delete users in the tenant but must not touch Azure resources. What '
          'do you assign?',
     'options': {'A': 'Azure RBAC Contributor on the subscription',
                 'B': 'Azure RBAC Owner on the resource group',
                 'C': 'The Entra ID User Administrator role',
                 'D': 'The Entra ID Global Administrator role'},
     'a': 'C',
     'why': 'Entra roles govern the *directory*; Azure RBAC roles govern *resources*. User Administrator '
            'is the least-privilege directory role for managing users and groups. Global Administrator '
            'would work and is exactly what least privilege tells you not to do.'},
]

MY_ANSWERS = {'Q1': 'C', 'Q2': 'B', 'Q3': 'B', 'Q4': 'B', 'Q5': 'B', 'Q6': 'C'}

score = 0
for q in QUIZ:
    mine = MY_ANSWERS.get(q['id'], '').strip().upper()
    ok = mine == q['a']
    score += ok
    print(f'{"PASS" if ok else "FAIL"}  {q["id"]}: {q["q"]}')
    for k, v in q['options'].items():
        mark = '<-- correct' if k == q['a'] else ''
        print(f'         {k}. {v} {mark}')
    print(f'         your answer: {mine or "(blank)"}')
    print(f'         why: {q["why"]}\n')
print(f'Score: {score}/{len(QUIZ)}')

assert {q['id'] for q in QUIZ} == set(MY_ANSWERS), 'every question needs an answer key entry'
assert all(q['a'] in q['options'] for q in QUIZ), 'an answer key points at an option that does not exist'


**Next**: [Notebook 3 — Identity Governance](03_identity_governance.ipynb)